<a href="https://colab.research.google.com/github/souro26/bayesian-a-b-testing/blob/main/examples/02_saas_pricing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import argonx  # noqa: F401
except ImportError:
    !pip install -q git+https://github.com/souro26/bayesian-a-b-testing.git

# Example 02 — SaaS Subscription Pricing & Early Stopping

A B2B SaaS company is testing a new 'Enterprise' pricing tier. 

### The Business Hypothesis
The current page (**Control**) focuses on monthly billing ($50/mo). The **Variant** focuses on annual billing ($480/yr), which effectively offers a 20% discount but locks users in for a year.

*   **Primary Metric:** Average Revenue Per User (ARPU).
*   **The Bayesian Advantage:** High-ticket B2B sales are sparse and right-skewed. A few large annual contracts can throw off a standard t-test. We use a **Log-Normal** model to handle this skewness gracefully and **Sequential Stopping** to call the winner as soon as we hit statistical certainty—saving weeks of waiting.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from argonx import Experiment
from argonx.sequential import StoppingChecker

# Set aesthetic style
plt.style.use("seaborn-v0_8-muted")
sns.set_theme(style="whitegrid")

## 1. Why Log-Normal? Understanding the Data Skew

In SaaS, most customers pay the base rate, but a 'tail' of customers might buy multiple seats or add-ons. Let's look at the distribution we're dealing with.

In [ ]:
np.random.seed(42)
N_sample = 1000

# Simulate skewed revenue data
rev_data = np.random.lognormal(mean=np.log(50), sigma=0.6, size=N_sample)

plt.figure(figsize=(10, 5))
sns.histplot(rev_data, kde=True, color="#2563EB", bins=50)
plt.title("Distribution of SaaS Revenue Per User", fontsize=14, fontweight="bold")
plt.xlabel("Revenue ($)", fontsize=12)
plt.axvline(
    np.mean(rev_data),
    color="red",
    linestyle="--",
    label=f"Mean: ${np.mean(rev_data):.2f}",
)
plt.axvline(
    np.median(rev_data),
    color="green",
    linestyle="--",
    label=f"Median: ${np.median(rev_data):.2f}",
)
plt.legend()
plt.show()

The long right tail pulls the mean away from the median. A standard Gaussian model would assume the noise is symmetric, leading to incorrect confidence intervals. The Log-Normal model 'expects' this tail.

## 2. Simulating the Sequential Traffic

We'll simulate 4 weeks of traffic. In reality, Variant B is 15% better. We want to see if we can stop before Week 4.

In [ ]:
def generate_revenue(n, median, sigma):
    return np.random.lognormal(mean=np.log(median), sigma=sigma, size=n)


week_traffic = 600
control_median = 50
variant_median = 57.5  # 15% lift
sigma = 0.6

weeks = []
for w in range(1, 5):
    c = generate_revenue(week_traffic, control_median, sigma)
    v = generate_revenue(week_traffic, variant_median, sigma)
    weeks.append(
        pd.DataFrame(
            {
                "variant": ["control"] * week_traffic + ["variant_b"] * week_traffic,
                "revenue": np.concatenate([c, v]),
                "week": w,
            }
        )
    )

full_data = pd.concat(weeks)

## 3. The Sequential Bayesian Decision Engine

We initialize a `StoppingChecker` with our business constraints. We are willing to tolerate a **$0.50 ARPU risk** to call the test early.

In [ ]:
checker = StoppingChecker(
    loss_threshold=0.50,  # Max $0.50 expected loss to stop
    prob_best_min=0.95,  # 95% certainty
    min_sample_size=1000,  # Don't even look before 1000 users
    burn_in_users=500,
)

print("Sequential Monitoring Log:")
print("-" * 30)

for w in range(1, 5):
    df_at_week = full_data[full_data["week"] <= w]

    exp = Experiment(
        data=df_at_week,
        variant_col="variant",
        primary_metric="revenue",
        model="lognormal",
        control="control",
    )

    result = exp.run(n_draws=3000)
    n_counts = df_at_week["variant"].value_counts().to_dict()

    status = checker.update(
        samples=result.samples,
        variant_names=["control", "variant_b"],
        control="control",
        n_users_per_variant=n_counts,
    )

    print(f"End of Week {w}: {status.recommendation}")
    if status.safe_to_stop:
        print(f"\n>>> STOPPING EARLY AT WEEK {w} <<<")
        break

## 4. Visualizing the Evidence Path

The trajectory plot shows how our expected loss dropped as more data arrived. Once it crosses the threshold (red dashed line), it's safe to ship.

In [ ]:
checker.plot_trajectory(
    figsize=(14, 8), suptitle="ARPU Experiment — Evidence Accumulation"
)

## Final Decision Summary

In [ ]:
result.plot(metric_name="Monthly ARPU ($)")

### Conclusion

By using **argonx**, the team was able to:
1.  **Correctly Model Revenue:** The Log-Normal model accounted for the high-variance nature of SaaS sales.
2.  **Save Time:** The experiment hit its 'Safe to Stop' criteria at **Week 2**. If they had waited for a standard 4-week fixed-horizon test, they would have wasted 2 weeks of engineering time. 
3.  **Quantify Risk:** Even at Week 2, we know that if we ship and we're wrong, the expected 'damage' to ARPU is less than $0.50, which is well within our business risk tolerance.